# Random Forest training


## Import project functions

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'core.py').exists() and (ROOT / 'FallDetection_Software/core.py').exists():
    ROOT = ROOT / 'FallDetection_Software'
assert (ROOT / 'core.py').exists(), 'Open this notebook in the FallDetection_Software folder.'
sys.path.insert(0, str(ROOT))
from core import FEATURES, POLICY
from train_random_forest import PLAN, prepare_development, fit_candidates, freeze_forest, evaluate_models
from random_forest import load_forest, predict_forest
print('Project located. The original application uses the decision tree.')
print('Features:', len(FEATURES))


Project located. The original application uses the decision tree.
Features: 12


## Read the fixed experiment plan

In [2]:
print(json.dumps(PLAN, indent=2))
print('Features:', ', '.join(FEATURES))

{
  "algorithm": "RandomForestClassifier",
  "n_estimators": 100,
  "max_depth_candidates": [
    5,
    8,
    null
  ],
  "min_samples_leaf_candidates": [
    2,
    5
  ],
  "threshold_candidates": [
    0.35,
    0.5,
    0.65
  ],
  "max_features": "sqrt",
  "class_weight": "balanced_subsample",
  "random_state": 42,
  "selection": "Validation F2, then precision, shallower depth, larger minimum leaf, higher threshold.",
  "split": {
    "train": [
      1,
      2,
      3
    ],
    "validation": [
      4
    ],
    "reused_test": [
      5,
      6
    ]
  },
  "comparison_status": "Exploratory comparison on previously inspected subjects 5-6. Not a new independent final test."
}
Features: acc_mean, acc_std, acc_min, acc_max, acc_range, gyro_mean, gyro_std, gyro_max, tilt_change, acc_last, gyro_last, acc_diff_max


## Load development windows

In [3]:
training, validation = prepare_development(ROOT)
xtrain, ytrain = training[2:]
xval, yval = validation[2:]
print('xtrain shape:', xtrain.shape)
print('Training fall windows:', int(ytrain.sum()))
print('Validation fall windows:', int(yval.sum()))


Training: 3237 windows; validation: 1147 windows.
xtrain shape: (3237, 12)
Training fall windows: 131
Validation fall windows: 30


## Train candidates and select using validation


In [4]:
clf, threshold, candidates = fit_candidates(training, validation)
print('Selected model:', type(clf).__name__)
print('Trees:', clf.n_estimators)
print('Depth:', clf.max_depth)
print('Minimum leaf size:', clf.min_samples_leaf)
print('Score threshold:', threshold)


Fitted 100 trees: depth=5, minimum leaf=2
Fitted 100 trees: depth=5, minimum leaf=5
Fitted 100 trees: depth=8, minimum leaf=2
Fitted 100 trees: depth=8, minimum leaf=5
Fitted 100 trees: depth=None, minimum leaf=2
Fitted 100 trees: depth=None, minimum leaf=5
Selected model: RandomForestClassifier
Trees: 100
Depth: 5
Minimum leaf size: 5
Score threshold: 0.65


## Freeze the selected forest

In [5]:
forest = freeze_forest(clf, threshold, candidates, ROOT)
print('Saved model/trained_forest.json')
print('Model ID:', forest['model_id'])

Selected settings frozen: depth=5, leaf=5, threshold=0.65
Saved model/trained_forest.json
Model ID: fall-forest-v1


## Compare actual predictions

In [6]:
report = evaluate_models(clf, forest, training, validation, ROOT)
comparison = pd.read_csv(ROOT / 'reports/random_forest_comparison.csv')
columns = ['method', 'accuracy', 'precision', 'recall', 'f1', 'false_event_groups_in_nonfall_recordings']
print(comparison.loc[comparison['split'].eq('reused_test'), columns].round(4).to_string(index=False))


Evaluated train: 3237 windows; portable scores match sklearn.
Evaluated validation: 1147 windows; portable scores match sklearn.
Evaluated reused_test: 2111 windows; portable scores match sklearn.
       method  accuracy  precision  recall    f1  false_event_groups_in_nonfall_recordings
decision_tree    0.9588     0.3740  0.8214 0.514                                        15
random_forest    0.9730     0.4921  0.5536 0.521                                         4


## Understand the tradeoff

In [7]:
for method, result in report['splits']['reused_test'].items():
    measured = result['window_metrics']
    (tn, fp), (fn, tp) = measured['confusion_matrix']
    print(f'{method}: {tp} fall windows detected; {fn} missed; {fp} false-positive windows.')
    print(f"  Balanced accuracy {measured['balanced_accuracy']:.2%}; F1 {measured['f1']:.3f}")


decision_tree: 46 fall windows detected; 10 missed; 77 false-positive windows.
  Balanced accuracy 89.20%; F1 0.514
random_forest: 31 fall windows detected; 25 missed; 32 false-positive windows.
  Balanced accuracy 76.90%; F1 0.521


## Try sensor inputs with the second model

In [8]:
loaded_forest = load_forest(ROOT / 'model/trained_forest.json')
for kind in ['walking', 'fall']:
    csv_path = ROOT / f'examples/{kind}_window.csv'
    predictions = predict_forest(loaded_forest, pd.read_csv(csv_path))
    print(kind, 'training excerpt:')
    for prediction in predictions:
        label = 'Suspected fall' if prediction['suspected_fall'] else 'No fall predicted'
        print(f"  {prediction['time_s']:.2f} s: {label}; score {prediction['fall_score']:.3f}")


walking training excerpt:
  1.02 s: No fall predicted; score 0.109
  1.27 s: No fall predicted; score 0.097
fall training excerpt:
  1.00 s: Suspected fall; score 0.971
  1.28 s: Suspected fall; score 0.972
